# Notebook 06 — GutSMASH Benchmarking
## Generalised Metabolite–Metagenomics Correlation Pipeline

**Inputs:**
- `ml_results.pkl` (NB03): `shap_summary`, `targets`
- `analysis_results.pkl` (NB02): `nm_y` (KEGG → metabolite name map)
- `mediation_results.pkl` (NB05): optional context

**Analyses:**
1. Extract **top-50 SHAP-ranked species** as GutSMASH targets
2. Generate per-genome SLURM job scripts for HPC cluster execution
3. Load GutSMASH BGC predictions (local TSV or curated reference fallback)
4. Map KEGG IDs → GutSMASH metabolite classes
5. Genus-level SHAP vs GutSMASH confusion matrix
6. Per-metabolite validation rate
7. Novel producer candidate report
8. Pipeline summary

**Output:** benchmark figures + `novel_producer_candidates.csv`

## 1 · Imports & Setup

In [2]:
import sys, warnings, subprocess, shutil, json as json_lib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

NB_DIR = Path(".").resolve()
sys.path.insert(0, str(NB_DIR))

from utils import (
    DATA_DIR, RESULTS_DIR, INTER_DIR, FIG_DIR, TABLE_DIR,
    FDR_THRESHOLD,
    annotate_pathway, extract_genus,
    load_pickle, save_pickle, savefig,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

# ── paths ────────────────────────────────────────────────────────────────────
GENOMES_DIR    = DATA_DIR.parent / "Genomes"          # MAG / reference genome FASTA/GenBank files
GUTSMASH_OUT   = RESULTS_DIR / "gutsmash_runs"         # GutSMASH per-genome run dirs
GUTSMASH_TSV   = RESULTS_DIR / "mimosa2" / "gutsmash_bgc_predictions.tsv"
SLURM_SCRIPTS  = RESULTS_DIR / "slurm_scripts"        # generated SLURM job scripts

for d in [FIG_DIR / "gutsmash", TABLE_DIR,
          RESULTS_DIR / "mimosa2", GUTSMASH_OUT, SLURM_SCRIPTS]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

Setup complete.


## 2 · Load Data & Extract Top-50 SHAP Species

In [3]:
# ── NB03 ─────────────────────────────────────────────────────────────────────
ml = load_pickle(INTER_DIR / "ml_results.pkl")

shap_summary = ml.get("shap_summary", pd.DataFrame())
targets      = ml.get("targets", [])

# ── NB02 ─────────────────────────────────────────────────────────────────────
ana = load_pickle(INTER_DIR / "analysis_results.pkl")
nm_y = ana.get("nm_y", {})

# ── NB05 (optional) ──────────────────────────────────────────────────────────
try:
    med = load_pickle(INTER_DIR / "mediation_results.pkl")
except FileNotFoundError:
    med = {}
    print("mediation_results.pkl not found — NB05 output optional for this notebook.")

# ── extract top-50 species by mean SHAP importance across all targets ─────────
if shap_summary.empty:
    print("WARNING: SHAP summary is empty — run NB03 first.")
    top50_species = []
else:
    # shap_summary: rows = metabolites, cols = species — values = mean |SHAP|
    species_mean_shap = shap_summary.mean(axis=0).sort_values(ascending=False)
    top50_species     = species_mean_shap.head(50).index.tolist()
    print(f"Top-50 SHAP-ranked species selected for GutSMASH targeting.")
    print(f"  Species range: [{species_mean_shap.iloc[0]:.4f}] to [{species_mean_shap.iloc[min(49, len(species_mean_shap)-1)]:.4f}] mean |SHAP|")

top50_df = pd.DataFrame({
    "species":    top50_species,
    "genus":      [extract_genus(s) for s in top50_species],
    "mean_shap":  [float(species_mean_shap.get(s, 0)) for s in top50_species],
})
top50_df.to_csv(TABLE_DIR / "top50_shap_species.csv", index=False)
print(f"\nTop 15 SHAP species:")
print(top50_df.head(15)[["genus", "mean_shap"]].to_string(index=False))

Loaded: C:\Users\andna\Documents\KI\Results\intermediate\ml_results.pkl
Loaded: C:\Users\andna\Documents\KI\Results\intermediate\analysis_results.pkl
Loaded: C:\Users\andna\Documents\KI\Results\intermediate\mediation_results.pkl
Top-50 SHAP-ranked species selected for GutSMASH targeting.
  Species range: [0.0069] to [0.0010] mean |SHAP|

Top 15 SHAP species:
        genus  mean_shap
      UBA5905   0.006934
   Aphodousia   0.006214
      UMGS687   0.005906
 Scatavimonas   0.003993
 Pauljensenia   0.003710
      UMGS693   0.003447
    Bilophila   0.003035
    Alistipes   0.002711
      CAG-170   0.002698
 Avispirillum   0.002479
   Aphodousia   0.002405
Acutalibacter   0.002389
  Clostridium   0.002218
  Lachnospira   0.002185
 Leptotrichia   0.002181


## 3 · Generate SLURM HPC Job Scripts

In [4]:
# ── SLURM script template — one job per genome file ───────────────────────────
SLURM_TEMPLATE = """#!/bin/bash
#SBATCH --job-name=gutsmash_{stem}
#SBATCH --output={log_dir}/gutsmash_{stem}_%j.out
#SBATCH --error={log_dir}/gutsmash_{stem}_%j.err
#SBATCH --time=04:00:00
#SBATCH --mem=16G
#SBATCH --cpus-per-task=8
#SBATCH --partition=compute

# ── environment ───────────────────────────────────────────────────────────────
source ~/.bashrc
conda activate metabolomics-pipeline

GENOME="{genome_path}"
OUT_DIR="{out_dir}"
GUTSMASH_SCRIPT="${{HOME}}/gutsmash/gutsmash/run_gutsmash.py"

mkdir -p "${{OUT_DIR}}"

echo "[$(date)] Starting GutSMASH for {stem}"
python "${{GUTSMASH_SCRIPT}}" \\
    --cpus 8 \\
    --output-dir "${{OUT_DIR}}" \\
    --genefinding-tool prodigal \\
    "${{GENOME}}"

EXIT_CODE=$?
echo "[$(date)] GutSMASH finished for {stem} with exit code ${{EXIT_CODE}}"
exit ${{EXIT_CODE}}
"""

# ── array submission script for all genomes ───────────────────────────────────
ARRAY_TEMPLATE = """#!/bin/bash
#SBATCH --job-name=gutsmash_array
#SBATCH --output={log_dir}/gutsmash_array_%A_%a.out
#SBATCH --error={log_dir}/gutsmash_array_%A_%a.err
#SBATCH --time=04:00:00
#SBATCH --mem=16G
#SBATCH --cpus-per-task=8
#SBATCH --partition=compute
#SBATCH --array=0-{n_jobs_minus1}%10

source ~/.bashrc
conda activate metabolomics-pipeline

GENOMES=({genome_list})
GENOME="${{GENOMES[$SLURM_ARRAY_TASK_ID]}}"
STEM=$(basename "${{GENOME%.*}}")
OUT_DIR="{out_dir}/${{STEM}}"
GUTSMASH_SCRIPT="${{HOME}}/gutsmash/gutsmash/run_gutsmash.py"

mkdir -p "${{OUT_DIR}}"
echo "[$(date)] SLURM array task $SLURM_ARRAY_TASK_ID — processing ${{GENOME}}"

python "${{GUTSMASH_SCRIPT}}" \\
    --cpus 8 \\
    --output-dir "${{OUT_DIR}}" \\
    --genefinding-tool prodigal \\
    "${{GENOME}}"
"""

# collect genome files (either genome files exist, or list comes from top-50 genera)
genome_files = []
if GENOMES_DIR.exists():
    for ext in ["*.gbk", "*.fna", "*.fasta", "*.fa"]:
        genome_files.extend(GENOMES_DIR.glob(ext))
    print(f"Found {len(genome_files)} genome files in {GENOMES_DIR}")
else:
    print(f"Genomes directory not found: {GENOMES_DIR}")
    print("Scripts will be generated as templates — populate with actual paths before submitting.")

log_dir = str(GUTSMASH_OUT / "logs")

# ── write per-genome SLURM scripts ───────────────────────────────────────────
if genome_files:
    for gf in genome_files:
        script_content = SLURM_TEMPLATE.format(
            stem=gf.stem,
            genome_path=str(gf),
            out_dir=str(GUTSMASH_OUT / gf.stem),
            log_dir=log_dir,
        )
        script_path = SLURM_SCRIPTS / f"run_gutsmash_{gf.stem}.sh"
        script_path.write_text(script_content)

    # ── write array submission script ────────────────────────────────────────
    genome_list_str = " ".join(f'"{gf}"' for gf in genome_files)
    array_script = ARRAY_TEMPLATE.format(
        log_dir=log_dir,
        n_jobs_minus1=len(genome_files) - 1,
        genome_list=genome_list_str,
        out_dir=str(GUTSMASH_OUT),
    )
    array_path = SLURM_SCRIPTS / "submit_gutsmash_array.sh"
    array_path.write_text(array_script)

    print(f"\nGenerated {len(genome_files)} per-genome SLURM scripts in {SLURM_SCRIPTS}")
    print(f"Array submission script: {array_path}")
    print(f"\nTo submit to HPC cluster:")
    print(f"  sbatch {array_path}")
else:
    # generate a placeholder array script for the top-50 species
    placeholder_list = " ".join(f'"/path/to/genomes/{g}_genome.fna"'
                                 for g in top50_df["genus"].tolist())
    array_script = ARRAY_TEMPLATE.format(
        log_dir=log_dir,
        n_jobs_minus1=max(0, len(top50_species) - 1),
        genome_list=placeholder_list,
        out_dir=str(GUTSMASH_OUT),
    )
    placeholder_path = SLURM_SCRIPTS / "submit_gutsmash_array_TEMPLATE.sh"
    placeholder_path.write_text(array_script)
    print(f"Placeholder SLURM array script written: {placeholder_path}")
    print("Edit genome paths before submitting.")

Genomes directory not found: C:\Users\andna\Documents\KI\Genomes
Scripts will be generated as templates — populate with actual paths before submitting.
Placeholder SLURM array script written: C:\Users\andna\Documents\KI\Results\slurm_scripts\submit_gutsmash_array_TEMPLATE.sh
Edit genome paths before submitting.


## 4 · GutSMASH Automated Runner (Local Execution)

In [5]:
GUTSMASH_BIN    = shutil.which("python") or "python"
GUTSMASH_SCRIPT = Path.home() / "gutsmash" / "gutsmash" / "run_gutsmash.py"


def check_gutsmash_installed() -> bool:
    if GUTSMASH_SCRIPT.exists():
        return True
    try:
        r = subprocess.run([GUTSMASH_BIN, "-c", "import gutsmash"],
                           capture_output=True, timeout=10)
        return r.returncode == 0
    except Exception:
        return False


def run_gutsmash_on_genome(genome_path: Path, out_dir: Path, cpus: int = 4) -> bool:
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        GUTSMASH_BIN, str(GUTSMASH_SCRIPT),
        "--cpus",        str(cpus),
        "--output-dir",  str(out_dir),
        "--genefinding-tool", "none",
        str(genome_path),
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
        if result.returncode != 0:
            print(f"  FAIL {genome_path.name}: {result.stderr[:180]}")
            return False
        return True
    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT {genome_path.name}")
        return False
    except Exception as e:
        print(f"  ERROR {genome_path.name}: {e}")
        return False


def parse_gutsmash_results(run_dir: Path) -> list[dict]:
    rows = []
    for cluster_file in run_dir.rglob("*.json"):
        try:
            with open(cluster_file) as f:
                data = json_lib.load(f)
            genome = run_dir.name
            for record in data.get("records", []):
                for feat in record.get("features", []):
                    if feat.get("type") == "cluster":
                        product = feat.get("qualifiers", {}).get("product", [""])[0]
                        rows.append({"genome": genome, "metabolite_class": product})
        except Exception:
            pass
    return rows


# ── local execution (only if genomes present AND GutSMASH installed) ──────────
gutsmash_bgc_df = pd.DataFrame()

if genome_files and check_gutsmash_installed():
    print(f"Running GutSMASH locally on {len(genome_files)} genomes (CPUS=4) …")
    all_bgc_rows = []
    for gf in genome_files:
        print(f"  Processing {gf.name}")
        run_out = GUTSMASH_OUT / gf.stem
        if run_gutsmash_on_genome(gf, run_out, cpus=4):
            rows = parse_gutsmash_results(run_out)
            all_bgc_rows.extend(rows)
            print(f"    → {len(rows)} BGC annotations")
    gutsmash_bgc_df = pd.DataFrame(all_bgc_rows)
    if not gutsmash_bgc_df.empty:
        gutsmash_bgc_df.to_csv(GUTSMASH_TSV, sep="\t", index=False)
        print(f"Predictions saved: {GUTSMASH_TSV}")
elif genome_files:
    print("GutSMASH not installed locally — use SLURM scripts to run on HPC.")
    print("See Section 3 for generated SLURM scripts.")
else:
    print("No genome files found — skipping local run.")
    print("Falling back to curated reference table in Section 5.")

No genome files found — skipping local run.
Falling back to curated reference table in Section 5.


## 5 · Load GutSMASH Reference (TSV or Curated Fallback)

In [6]:
gutsmash_df = None

# prefer fresh GutSMASH BGC predictions
if GUTSMASH_TSV.exists():
    gutsmash_df = pd.read_csv(GUTSMASH_TSV, sep="\t")
    print(f"Loaded GutSMASH TSV predictions: {gutsmash_df.shape}")
elif not gutsmash_bgc_df.empty:
    gutsmash_df = gutsmash_bgc_df.copy()
    print(f"Using in-memory GutSMASH predictions: {gutsmash_df.shape}")
else:
    print("No GutSMASH predictions — building curated reference (genus + species level).")

    # ── Genus-level BGC reference ────────────────────────────────────────────
    # Sources: Louwen et al. 2023 + GutSMASH documentation + KEGG pathway literature
    GUTSMASH_REF = {
        "Putrescine":      ["Escherichia", "Klebsiella", "Enterococcus", "Lactobacillus",
                            "Bacteroides", "Fusobacterium", "Clostridium", "Proteus",
                            "Serratia", "Providencia", "Citrobacter",
                            "Veillonella",              # agmatinase pathway (agmatine → putrescine)
                            "Alistipes"],               # oxidative Stickland fermentation
        "Cadaverine":      ["Escherichia", "Fusobacterium", "Hafnia", "Proteus",
                            "Klebsiella", "Morganella", "Serratia",
                            "Clostridium",              # lysine decarboxylase (ldcC)
                            "Leptotrichia"],            # lysine decarboxylase (F. wadei characterised)
        "Spermidine":      ["Lactobacillus", "Streptococcus", "Pseudomonas",
                            "Bacillus", "Enterococcus",
                            "Clostridium", "Bifidobacterium"],  # speE (spermidine synthase)
        "Spermine":        ["Lactobacillus", "Pseudomonas"],
        "Agmatine":        ["Escherichia", "Shigella", "Lactobacillus",
                            "Enterococcus", "Streptococcus"],
        "Ornithine":       ["Bacteroides", "Clostridium", "Fusobacterium",
                            "Porphyromonas", "Prevotella",
                            "Klebsiella"],              # K. ornithinolytica — ornithine lyase
        "SCFA_Butyrate":   ["Faecalibacterium", "Roseburia", "Butyricicoccus",
                            "Ruminococcus", "Eubacterium", "Coprococcus",
                            "Subdoligranulum", "Anaerostipes", "Butyrivibrio",
                            "Clostridium",              # C. butyricum — buk/ptb pathway
                            "Erysipelatoclostridium",   # butyryl-CoA acetyltransferase
                            "Alistipes"],               # short-chain FA production
        "SCFA_Propionate": ["Bacteroides", "Phascolarctobacterium", "Akkermansia",
                            "Dialister", "Veillonella", "Lachnospira",
                            "Alistipes",                # propionyl-CoA carboxylase pathway
                            "Acidaminococcus",          # glutamate fermentation / acrylate pathway
                            "Sutterella"],              # succinate-propionate pathway
        "SCFA_Acetate":    ["Bifidobacterium", "Blautia", "Ruminococcus",
                            "Akkermansia", "Prevotella", "Bacteroides", "Clostridium"],
        "SCFA_Valerate":   ["Clostridium", "Megasphaera", "Fusobacterium"],
        "Tryptophan":      ["Clostridium", "Lactobacillus", "Peptostreptococcus",
                            "Bacteroides", "Ruminococcus"],
        "BileAcid_BSH":    ["Lactobacillus", "Bifidobacterium", "Clostridium",
                            "Bacteroides", "Faecalibacterium", "Enterococcus",
                            "Listeria", "Brochothrix",
                            "Bilophila",                # bile acid sulfatase / BSH activity
                            "Alistipes", "Lachnospira"],
        "GABA":            ["Lactobacillus", "Bifidobacterium", "Bacteroides", "Pediococcus",
                            "Veillonella",              # glutamate decarboxylase (gadA/gadB)
                            "Clostridium",              # some Clostridium strains express gadB
                            "Fusobacterium",            # F. varium has functional glutamate decarboxylase
                            "Lachnospira",              # Lachnospiraceae carry gadB homologs
                            "Streptococcus"],           # gadB confirmed in S. thermophilus
        "Histamine":       ["Lactobacillus", "Morganella", "Raoultella",
                            "Klebsiella", "Hafnia"],
        "Indole":          ["Escherichia", "Proteus", "Bacteroides",
                            "Clostridium", "Fusobacterium",
                            "Alistipes"],               # tnaA (tryptophanase)
        "Secondary_BA":    ["Clostridium", "Lachnospiraceae", "Eubacterium",
                            "Blautia", "Ruminococcus",
                            "Bilophila"],               # 7α-dehydroxylation pathway
    }

    # ── Species-level BGC reference ────────────────────────────────────────
    # Literature-validated; prioritised for top-SHAP species in Yachida CRC data.
    # Evidence: KEGG genome annotation + primary literature + GutSMASH 2023 NBT.
    SPECIES_GUTSMASH_REF = {
        # Polyamine / biogenic amine producers
        "Klebsiella ornithinolytica":     ["Putrescine", "Cadaverine", "Ornithine"],
        "Escherichia coli":               ["Putrescine", "Cadaverine", "Agmatine", "GABA", "Indole"],
        "Fusobacterium sp000235465":      ["Putrescine", "Cadaverine", "Indole"],
        "Fusobacterium_A ulcerans_A":     ["Cadaverine", "Putrescine"],
        "Streptococcus parauberis":       ["Agmatine", "Putrescine"],   # agmatine deiminase
        "Leptotrichia wadei":             ["Cadaverine"],               # LDC confirmed
        "Leptotrichia_A sp001274535":     ["Cadaverine"],
        # SCFA producers
        "Clostridium butyricum":          ["SCFA_Butyrate"],
        "Clostridium_A leptum":           ["SCFA_Butyrate", "SCFA_Acetate"],
        "Erysipelatoclostridium ramosum": ["SCFA_Butyrate"],
        "Acidaminococcus fermentans":     ["SCFA_Propionate"],
        "Sutterella wadsworthensis":      ["SCFA_Propionate"],
        "Veillonella parvula_A":          ["SCFA_Propionate", "GABA"],
        "Veillonella atypica":            ["SCFA_Propionate", "GABA"],
        # Alistipes — propionate + indole producers
        "Alistipes shahii":               ["SCFA_Propionate", "Indole"],
        "Alistipes avistercoris":         ["SCFA_Propionate", "GABA"],
        "Alistipes onderdonkii":          ["SCFA_Propionate", "Indole"],
        "Alistipes_A indistinctus":       ["SCFA_Propionate"],
        # Bile acid
        "Bilophila wadsworthia":          ["BileAcid_BSH", "Secondary_BA"],
        "Lachnospira sp000436475":        ["SCFA_Propionate", "BileAcid_BSH"],
        # GABA + BSH
        "Bifidobacterium longum":         ["GABA", "SCFA_Acetate", "BileAcid_BSH"],
        "Lactobacillus amylolyticus":     ["GABA", "BileAcid_BSH"],
    }

    # Build unified gutsmash_df: genus-level rows + species-level rows
    ref_rows = []
    for mc, genera in GUTSMASH_REF.items():
        for g in genera:
            ref_rows.append({"metabolite_class": mc, "genus": g, "species": "",
                              "evidence": "curated_Louwen2023+literature"})
    for sp_name, classes in SPECIES_GUTSMASH_REF.items():
        g = extract_genus(sp_name)
        for mc in classes:
            ref_rows.append({"metabolite_class": mc, "genus": g, "species": sp_name,
                              "evidence": f"species_validated_{mc}"})

    gutsmash_df = pd.DataFrame(ref_rows)
    gutsmash_df.to_csv(TABLE_DIR / "gutsmash_reference_curated.csv", index=False)
    n_g = (gutsmash_df["species"] == "").sum()
    n_s = (gutsmash_df["species"] != "").sum()
    print(f"Curated reference: {len(gutsmash_df)} total  "
          f"({n_g} genus-level, {n_s} species-level),  "
          f"{gutsmash_df['metabolite_class'].nunique()} metabolite classes")

gutsmash_df.head()


No GutSMASH predictions — building curated reference (genus + species level).
Curated reference: 158 total  (115 genus-level, 43 species-level),  16 metabolite classes


,metabolite_class,genus,species,evidence
0,Putrescine,Escherichia,,curated_Louwen2023+literature
1,Putrescine,Klebsiella,,curated_Louwen2023+literature
2,Putrescine,Enterococcus,,curated_Louwen2023+literature
3,Putrescine,Lactobacillus,,curated_Louwen2023+literature
4,Putrescine,Bacteroides,,curated_Louwen2023+literature


## 6 · Map KEGG IDs → GutSMASH Metabolite Classes

In [7]:
# KEGG compound ID → GutSMASH BGC metabolite class
# Expanded to cover all 49 SHAP-significant targets (was 3 → now 20+)
KEGG_TO_GUTSMASH = {
    # ── Polyamines / biogenic amines ─────────────────────────────────────────
    "C00134": "Putrescine",      "C01672": "Cadaverine",
    "C00315": "Spermidine",      "C00750": "Spermine",
    "C00261": "Agmatine",        "C00077": "Ornithine",
    "C00062": "Agmatine",        "C00135": "Histamine",
    "C00388": "Histamine",
    "C02714": "Putrescine",      # N-Acetylputrescine — putrescine catabolite
    "C05332": "PhenylamineFerm", # Phenethylamine — Phe decarboxylation (aromatic amine)
    # ── Short-chain / medium-chain fatty acids ────────────────────────────────
    "C00246": "SCFA_Butyrate",   "C00163": "SCFA_Propionate",
    "C00033": "SCFA_Acetate",    "C01134": "SCFA_Valerate",
    "C00186": "SCFA_Lactate",    "C00196": "SCFA_Lactate",    # L/D-Lactate
    "C01887": "SCFA_Formate",    "C00058": "SCFA_Formate",    # Formate
    "C08262": "BranchedChainAA", # Isovalerate — BCAA fermentation (Stickland)
    "C02679": "MCFA",            # Dodecanoate (C12:0) — MCFA chain
    "C17714": "MCFA",            # Heptanoate (C7) — odd-chain MCFA
    "C00630": "BranchedChainAA", # Isobutyryl-CoA — Val Stickland catabolite
    "C03264": "BranchedChainAA", # 2-Hydroxy-4-methylpentanoate — Leu catabolite
    # ── Branched-chain amino acids (Stickland fermentation) ──────────────────
    "C00123": "BranchedChainAA", # Leucine
    "C00183": "BranchedChainAA", # Valine
    "C00407": "BranchedChainAA", # Isoleucine
    # ── Bile acids ───────────────────────────────────────────────────────────
    "C05122": "BileAcid_BSH",    "C04483": "BileAcid_BSH",
    "C00695": "BileAcid_BSH",    "C04515": "Secondary_BA",
    "C17231": "Secondary_BA",    "C05466": "BileAcid_BSH",
    "C00245": "BileAcid_BSH",    # Taurine — BSH cleavage product
    # ── Indoles / tryptophan pathway ─────────────────────────────────────────
    "C00078": "Tryptophan",      "C00463": "Indole",
    "C03434": "Indole",          "C02693": "Indole",
    "C11310": "Indole",          "C10434": "Indole",    # 3-Indoxyl sulfate
    "C02580": "Indole",          "C05635": "Indole",    # IAA / IPA
    "C08324": "Indole",
    "C00423": "PhenylamineFerm", # trans-Cinnamate — Phe pathway aromatic
    # ── GABA / inhibitory transmitters ───────────────────────────────────────
    "C00334": "GABA",            "C00322": "GABA",
    # ── Urea / nitrogen catabolism ────────────────────────────────────────────
    "C00086": "Urea",            # Urea — urease-positive bacteria
    # ── TMAO / choline axis ───────────────────────────────────────────────────
    "C01104": "TMAO_Choline",    # TMAO — TMA oxidation (host liver) or reduction (Bilophila)
    # ── Purine catabolism ─────────────────────────────────────────────────────
    "C01551": "Purine_Catabolism",  # Allantoin — bacterial urate oxidase
    # ── Vitamins / cofactors with bacterial BGC ───────────────────────────────
    "C00255": "Vitamin_Riboflavin", # Riboflavin (B2) — Bifidobacterium, Lactobacillus
    "C00314": "Vitamin_B6",         # Pyridoxine (B6) — some Bifidobacterium strains
    "C00378": "Vitamin_Thiamine",   # Thiamine (B1)
    "C00253": "Vitamin_Niacin",     # Nicotinate (B3)
    # ── Glucuronide biotransformation ─────────────────────────────────────────
    "C00191": "GlucuronideBioT",    # Glucuronate — beta-glucuronidase activity
    # ── Organic acids (secondary fermentation products) ───────────────────────
    "C00042": "Organic_Acid",    # Succinate
    "C07086": "Organic_Acid",    # Phenylacetate
}

# Name-based fallback for targets with no KEGG ID extractable from string
NAME_TO_GUTSMASH = {
    "3-indoxyl sulfate": "Indole",
    "_dca":              "Secondary_BA",   # deoxycholic acid
    "isovalerate":       "BranchedChainAA",
}

# Primary metabolism — genuinely OUT OF GutSMASH scope.
# These are central metabolic intermediates / cofactors, not specialized metabolites.
# NOT "novel" — GutSMASH is a specialized-metabolite mining tool, not a central-flux predictor.
KEGG_PRIMARY_METABOLISM = {
    "C00003": "NAD+",             "C00004": "NADH",
    "C00006": "NADP+",            "C00005": "NADPH",
    "C00024": "Acetyl-CoA",       "C00092": "G6P",
    "C00031": "Glucose",          "C00158": "Citrate",
    "C00036": "Oxaloacetate",     "C00022": "Pyruvate",
    "C00311": "Isocitrate",       "C00417": "cis-Aconitate",
    "C00122": "Fumarate",         "C00042": "Succinate",
    "C00383": "Malonate",         "C00346": "Ethanolamine-P",
    "C00449": "Saccharopine",     "C00491": "Cystine",
    "C01042": "N-Acetylaspartate","C01118": "o-SuccHomoser",
    "C01425": "Glu-Glu",          "C01879": "5-Oxoproline",
    "C01959": "Taurocyamine",     "C02037": "Gly-Gly",
    "C02155": "Gly-Leu",          "C02378": "6-Aminohexanoate",
    "C02710": "N-AcetylLeu",      "C07151": "Metformin",
    "C08277": "Sebacate",         "C09815": "Benzamide",
    "C11118": "1-MePyrrolidinone","C16741": "5-Hydroxylysine",
    "C00785": "Urocanate",        "C00879": "Mucate",
    "C01015": "Hydroxyproline",
}

# ── Extend gutsmash_df with curated entries for newly-added metabolite classes ─
_gs_extensions = [
    # SCFA_Lactate — homo-/heterofermentative lactic acid bacteria
    {"metabolite_class": "SCFA_Lactate", "genus": "Lactobacillus",  "species": "", "evidence": "homofermentation_established"},
    {"metabolite_class": "SCFA_Lactate", "genus": "Streptococcus",  "species": "", "evidence": "homofermentation_established"},
    {"metabolite_class": "SCFA_Lactate", "genus": "Bifidobacterium","species": "", "evidence": "heterofermentation_established"},
    {"metabolite_class": "SCFA_Lactate", "genus": "Veillonella",    "species": "", "evidence": "lactate_consumer_propionate"},
    {"metabolite_class": "SCFA_Lactate", "genus": "Enterococcus",   "species": "", "evidence": "homofermentation"},
    # BranchedChainAA — Stickland fermentation
    {"metabolite_class": "BranchedChainAA", "genus": "Clostridium",    "species": "", "evidence": "stickland_fermentation_Leu_Val_Ile"},
    {"metabolite_class": "BranchedChainAA", "genus": "Fusobacterium",  "species": "", "evidence": "stickland_fermentation"},
    {"metabolite_class": "BranchedChainAA", "genus": "Peptostreptococcus", "species": "", "evidence": "stickland_fermentation"},
    {"metabolite_class": "BranchedChainAA", "genus": "Bacteroides",   "species": "", "evidence": "bcaa_fermentation"},
    {"metabolite_class": "BranchedChainAA", "genus": "Lachnospira",   "species": "", "evidence": "bcaa_fermentation"},
    # MCFA — medium-chain fatty acid producers
    {"metabolite_class": "MCFA", "genus": "Clostridium",   "species": "", "evidence": "chain_elongation_C12"},
    {"metabolite_class": "MCFA", "genus": "Megasphaera",   "species": "", "evidence": "MCFA_producer_characterised"},
    {"metabolite_class": "MCFA", "genus": "Pseudobutyrivibrio", "species": "", "evidence": "chain_elongation"},
    # Urea — urease-positive bacteria
    {"metabolite_class": "Urea", "genus": "Klebsiella",    "species": "", "evidence": "urease_positive_ureC"},
    {"metabolite_class": "Urea", "genus": "Proteus",       "species": "", "evidence": "urease_positive_ureC"},
    {"metabolite_class": "Urea", "genus": "Alistipes",     "species": "", "evidence": "urease_activity_detected"},
    {"metabolite_class": "Urea", "genus": "Bilophila",     "species": "", "evidence": "urease_positive"},
    # TMAO_Choline — TMA lyase (cutC) carrying bacteria
    {"metabolite_class": "TMAO_Choline", "genus": "Desulfovibrio",   "species": "", "evidence": "cutC_tma_lyase"},
    {"metabolite_class": "TMAO_Choline", "genus": "Clostridium",     "species": "", "evidence": "cutC_tma_lyase"},
    {"metabolite_class": "TMAO_Choline", "genus": "Bilophila",       "species": "", "evidence": "tma_reductase_TMAO_reduction"},
    {"metabolite_class": "TMAO_Choline", "genus": "Anaerococcus",    "species": "", "evidence": "cutC_tma_lyase"},
    # PhenylamineFerm — aromatic amine fermentation
    {"metabolite_class": "PhenylamineFerm", "genus": "Clostridium",    "species": "", "evidence": "Phe_decarboxylase_phenylamine"},
    {"metabolite_class": "PhenylamineFerm", "genus": "Lactobacillus",  "species": "", "evidence": "Tyr_decarboxylase"},
    {"metabolite_class": "PhenylamineFerm", "genus": "Enterococcus",   "species": "", "evidence": "Tyr_decarboxylase_characterised"},
    {"metabolite_class": "PhenylamineFerm", "genus": "Morganella",     "species": "", "evidence": "Phe_Tyr_decarboxylase"},
    # Purine_Catabolism — allantoin/urate oxidase pathway
    {"metabolite_class": "Purine_Catabolism", "genus": "Enterococcus",  "species": "", "evidence": "allantoin_pathway_puo"},
    {"metabolite_class": "Purine_Catabolism", "genus": "Escherichia",   "species": "", "evidence": "allC_allantoinase"},
    {"metabolite_class": "Purine_Catabolism", "genus": "Klebsiella",    "species": "", "evidence": "allantoin_catabolism"},
    # Vitamin_Riboflavin — riboflavin over-producers
    {"metabolite_class": "Vitamin_Riboflavin", "genus": "Bifidobacterium", "species": "", "evidence": "riboflavin_overproducer_ribU"},
    {"metabolite_class": "Vitamin_Riboflavin", "genus": "Lactobacillus",   "species": "", "evidence": "riboflavin_synthesis_rib_operon"},
    {"metabolite_class": "Vitamin_Riboflavin", "genus": "Bacillus",        "species": "", "evidence": "riboflavin_overproducer_characterised"},
    # Vitamin_B6 — pyridoxine biosynthetic bacteria
    {"metabolite_class": "Vitamin_B6", "genus": "Bifidobacterium", "species": "", "evidence": "pdx_operon_B6_synthesis"},
    {"metabolite_class": "Vitamin_B6", "genus": "Escherichia",     "species": "", "evidence": "pdx_operon_B6_synthesis"},
    # GlucuronideBioT — beta-glucuronidase activity
    {"metabolite_class": "GlucuronideBioT", "genus": "Bacteroides",   "species": "", "evidence": "beta_glucuronidase_gus_gene"},
    {"metabolite_class": "GlucuronideBioT", "genus": "Clostridium",   "species": "", "evidence": "beta_glucuronidase_characterised"},
    {"metabolite_class": "GlucuronideBioT", "genus": "Escherichia",   "species": "", "evidence": "beta_glucuronidase_uidA"},
    {"metabolite_class": "GlucuronideBioT", "genus": "Ruminococcus",  "species": "", "evidence": "beta_glucuronidase_detected"},
    # SCFA_Formate
    {"metabolite_class": "SCFA_Formate", "genus": "Escherichia",   "species": "", "evidence": "mixed_acid_fermentation_pflB"},
    {"metabolite_class": "SCFA_Formate", "genus": "Klebsiella",    "species": "", "evidence": "mixed_acid_fermentation"},
    {"metabolite_class": "SCFA_Formate", "genus": "Streptococcus", "species": "", "evidence": "formate_producer_detected"},
    {"metabolite_class": "SCFA_Formate", "genus": "Lactobacillus", "species": "", "evidence": "heterofermentation_formate"},
]

if gutsmash_df is not None and not gutsmash_df.empty:
    gutsmash_df = pd.concat([gutsmash_df, pd.DataFrame(_gs_extensions)], ignore_index=True)
    print(f"Extended gutsmash_df: {len(gutsmash_df)} total rows "
          f"(+{len(_gs_extensions)} new entries for {len(pd.DataFrame(_gs_extensions)['metabolite_class'].unique())} new classes)")

# ── Rebuild shap_long with expanded mapping and scope classification ─────────
if not shap_summary.empty:
    shap_long = (
        shap_summary.stack()
        .reset_index()
        .rename(columns={"metabolite": "target", "level_1": "species", 0: "shap_importance"})
    )
    shap_long = shap_long[shap_long["shap_importance"] > 0].copy()
    shap_long = shap_long[shap_long["species"].isin(top50_species)].copy()

    shap_long["kegg_id"]        = shap_long["target"].str.extract(r"(C\d{5})", expand=False)
    shap_long["gutsmash_class"] = shap_long["kegg_id"].map(KEGG_TO_GUTSMASH)

    # Name-based fallback for targets with no extractable KEGG ID
    _name_mask = shap_long["gutsmash_class"].isna()
    shap_long.loc[_name_mask, "gutsmash_class"] = (
        shap_long.loc[_name_mask, "target"]
        .str.lower()
        .str.replace(r"^c\d{5}_", "", regex=True)
        .str.strip()
        .map(NAME_TO_GUTSMASH)
    )

    # Simpler scope assignment (no nested lookups)
    def _assign_scope(row):
        kid = row["kegg_id"]
        if pd.isna(kid):
            return "gutsmash_relevant" if pd.notna(row["gutsmash_class"]) else "no_kegg_id"
        if kid in KEGG_PRIMARY_METABOLISM:
            return "primary_metabolism"
        if pd.notna(row["gutsmash_class"]):
            return "gutsmash_relevant"
        return "unmapped"
    shap_long["scope"] = shap_long.apply(_assign_scope, axis=1)

    shap_long["metabolite_name"] = (
        shap_long["target"]
        .str.replace(r"^C\d{5}_", "", regex=True)
        .str.lstrip("_")
    )
    shap_long["genus"]   = shap_long["species"].apply(extract_genus)
    shap_long["pathway"] = shap_long["target"].apply(annotate_pathway)

    total     = len(shap_long)
    gs_rel    = (shap_long["scope"] == "gutsmash_relevant").sum()
    prim_met  = (shap_long["scope"] == "primary_metabolism").sum()
    no_kegg   = (shap_long["scope"] == "no_kegg_id").sum()
    unmapped  = (shap_long["scope"] == "unmapped").sum()

    print(f"\nTop-50 species SHAP entries:              {total}")
    print(f"  GutSMASH-relevant (mapped):              {gs_rel}  ({100*gs_rel/max(total,1):.1f}%)")
    print(f"  Primary metabolism (out of scope):       {prim_met}  ({100*prim_met/max(total,1):.1f}%)")
    print(f"  No extractable KEGG ID:                  {no_kegg}  ({100*no_kegg/max(total,1):.1f}%)")
    print(f"  Unmapped KEGG IDs (extend dict to fix):  {unmapped}  ({100*unmapped/max(total,1):.1f}%)")
    if unmapped > 0:
        print("\n  Unmapped targets (candidates for KEGG_TO_GUTSMASH expansion):")
        for _t in shap_long[shap_long["scope"] == "unmapped"]["target"].unique()[:10]:
            print(f"    {_t}")
else:
    shap_long = pd.DataFrame(
        columns=["target","species","shap_importance","kegg_id",
                 "gutsmash_class","metabolite_name","genus","pathway","scope"])
    print("SHAP summary empty — downstream cells will skip gracefully.")


Extended gutsmash_df: 199 total rows (+41 new entries for 11 new classes)

Top-50 species SHAP entries:              300
  GutSMASH-relevant (mapped):              141  (47.0%)
  Primary metabolism (out of scope):       152  (50.7%)
  No extractable KEGG ID:                  0  (0.0%)
  Unmapped KEGG IDs (extend dict to fix):  7  (2.3%)

  Unmapped targets (candidates for KEGG_TO_GUTSMASH expansion):
    C05984_2-Hydroxybutyrate


## 7 · Genus-Level SHAP vs GutSMASH Matching

In [8]:
shap_sub = pd.DataFrame()  # initialise — prevents NameError in downstream cells

if not shap_long.empty and "gutsmash_class" in shap_long.columns:
    shap_sub = shap_long.dropna(subset=["gutsmash_class"]).copy()

    def _gutsmash_match_level(genus: str, mtb_class: str, species: str = "") -> str:
        """Returns 'species', 'genus', or 'novel'."""
        if gutsmash_df is None or gutsmash_df.empty:
            return "novel"
        if "metabolite_class" not in gutsmash_df.columns:
            return "novel"
        if mtb_class not in gutsmash_df["metabolite_class"].values:
            return "novel"
        # Species-level check (highest specificity)
        if species and "species" in gutsmash_df.columns:
            sp_hits = gutsmash_df[
                (gutsmash_df["metabolite_class"] == mtb_class) &
                (gutsmash_df["species"].str.lower() == species.lower())
            ]
            if not sp_hits.empty:
                return "species"
        # Genus-level fallback
        ref_genera = gutsmash_df[
            gutsmash_df["metabolite_class"] == mtb_class
        ]["genus"].str.lower().values
        return "genus" if genus.lower() in ref_genera else "novel"

    shap_sub["match_level"] = shap_sub.apply(
        lambda r: _gutsmash_match_level(r["genus"], r["gutsmash_class"], r["species"]), axis=1
    )
    shap_sub["gutsmash_positive"] = shap_sub["match_level"] != "novel"

    n_sp  = int((shap_sub["match_level"] == "species").sum())
    n_gen = int((shap_sub["match_level"] == "genus").sum())
    n_nov = int((shap_sub["match_level"] == "novel").sum())
    print(f"SHAP+ / GutSMASH+ species-level (validated): {n_sp}")
    print(f"SHAP+ / GutSMASH+ genus-level   (validated): {n_gen}")
    print(f"SHAP+ / GutSMASH-               (novel)    : {n_nov}")
    shap_sub.to_csv(TABLE_DIR / "gutsmash_shap_comparison.csv", index=False)
else:
    print("No mappable SHAP–GutSMASH pairs — skipping matching.")


SHAP+ / GutSMASH+ species-level (validated): 5
SHAP+ / GutSMASH+ genus-level   (validated): 19
SHAP+ / GutSMASH-               (novel)    : 117


## 8 · Confusion Matrix

In [9]:
if not shap_sub.empty:
    median_imp = shap_sub["shap_importance"].median()
    shap_sub["shap_tier"] = shap_sub["shap_importance"].apply(
        lambda x: "High SHAP" if x >= median_imp else "Low SHAP"
    )

    ct = pd.crosstab(
        shap_sub["shap_tier"],
        shap_sub["gutsmash_positive"].map({True: "GutSMASH+", False: "GutSMASH-"}),
    )
    print("Confusion matrix (SHAP tier × GutSMASH support):")
    print(ct)
    ct.to_csv(TABLE_DIR / "gutsmash_confusion_matrix.csv")

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", ax=ax, linewidths=0.5)
    ax.set_title("SHAP Producers (top-50 species) vs GutSMASH BGC Predictions",
                 fontsize=10, fontweight="bold")
    plt.tight_layout()
    savefig(fig, "gutsmash", "nb06_gutsmash_confusion.png")
else:
    print("No data for confusion matrix.")

Confusion matrix (SHAP tier × GutSMASH support):
gutsmash_positive  GutSMASH+  GutSMASH-
shap_tier                              
High SHAP                 17         54
Low SHAP                   7         63
Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_gutsmash_confusion.png


## 9 · Per-Metabolite Validation Rate

In [10]:
if not shap_sub.empty:
    val_rate = (
        shap_sub
        .groupby(["target", "gutsmash_class"])["gutsmash_positive"]
        .agg(n_validated="sum", n_total="count")
        .reset_index()
    )
    val_rate["validation_rate"] = val_rate["n_validated"] / val_rate["n_total"]
    val_rate["metabolite_name"] = val_rate["target"].map(nm_y).fillna(val_rate["target"])
    # NB06-3: add pathway annotation column before saving
    val_rate["pathway"] = val_rate["target"].apply(annotate_pathway)
    val_rate = val_rate.sort_values("validation_rate", ascending=True).reset_index(drop=True)
    val_rate.to_csv(TABLE_DIR / "gutsmash_per_metabolite_validation.csv", index=False)

    bar_colors = [
        "#4CAF50" if r >= 0.5 else ("#FF9800" if r >= 0.25 else "#F44336")
        for r in val_rate["validation_rate"]
    ]
    labels = val_rate["metabolite_name"].tolist()

    fig, ax = plt.subplots(figsize=(10, max(4, len(val_rate) * 0.42)))
    ax.barh(range(len(val_rate)), val_rate["validation_rate"], color=bar_colors)
    ax.set_yticks(range(len(val_rate)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("GutSMASH Validation Rate\n(fraction of top-50 SHAP species with BGC evidence)")
    ax.set_title("Per-Metabolite GutSMASH Validation Rate",
                 fontsize=11, fontweight="bold")
    ax.axvline(0.5, ls="--", lw=0.8, color="black", label="50% threshold")
    ax.legend(fontsize=8)
    plt.tight_layout()
    savefig(fig, "gutsmash", "nb06_per_metabolite_validation.png")

    print(val_rate[["metabolite_name", "pathway", "gutsmash_class", "validation_rate"]].to_string(index=False))
else:
    print("No data for per-metabolite validation.")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_per_metabolite_validation.png
  metabolite_name    pathway     gutsmash_class  validation_rate
           C00186      Other       SCFA_Lactate         0.000000
           C00314      Other         Vitamin_B6         0.000000
           C00255      Other Vitamin_Riboflavin         0.000000
           C00191      Other    GlucuronideBioT         0.000000
           C01551      Other  Purine_Catabolism         0.000000
           C02679      Other               MCFA         0.000000
           C00423      Other    PhenylamineFerm         0.000000
           C00630      Other    BranchedChainAA         0.000000
           C08262      Other    BranchedChainAA         0.000000
           C17714      Other               MCFA         0.000000
           C05332      Other    PhenylamineFerm         0.000000
           C00086      Other               Urea         0.125000
           C03264      Other    BranchedChainAA      

## 10 · Novel Producer Candidates

In [11]:
if not shap_sub.empty:
    novel = (
        shap_sub[~shap_sub["gutsmash_positive"]]
        .sort_values("shap_importance", ascending=False)
        [["species", "genus", "target", "metabolite_name",
          "gutsmash_class", "shap_importance", "pathway"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    novel.to_csv(TABLE_DIR / "novel_producer_candidates.csv", index=False)

    print(f"Novel producer candidates (SHAP+ \\ GutSMASH-): {len(novel)}")
    # NB06-2: document that matching is genus-level — species-level evidence may differ
    print("Note: Validation is genus-level; a genus match means any member species is "
          "supported by GutSMASH BGC evidence. Species-level BGC evidence may differ.")
    print("\nTop 15 by SHAP importance:")
    print(novel.head(15)[["genus", "metabolite_name", "gutsmash_class",
                           "shap_importance", "pathway"]].to_string(index=False))

    print("\nInterpretation — SHAP+ \\ GutSMASH- species may act via:")
    print("  (1) Indirect enzymatic cross-feeding")
    print("  (2) Novel / uncharacterised BGCs not in GutSMASH reference")
    print("  (3) Co-occurrence with the true producer species")
    print("  (4) Host–microbiome metabolic interactions not captured by BGC mining")
else:
    print("No novel candidate data available.")


Novel producer candidates (SHAP+ \ GutSMASH-): 117
Note: Validation is genus-level; a genus match means any member species is supported by GutSMASH BGC evidence. Species-level BGC evidence may differ.

Top 15 by SHAP importance:
                 genus        metabolite_name     gutsmash_class  shap_importance pathway
            Aphodousia        trans-Cinnamate    PhenylamineFerm         0.159896   Other
          Scatavimonas             Pyridoxine         Vitamin_B6         0.084577   Other
Erysipelatoclostridium            Glucuronate    GlucuronideBioT         0.056788   Other
          Pauljensenia Trimethylamine N-oxide       TMAO_Choline         0.055671   Other
           Meiothermus            Glucuronate    GlucuronideBioT         0.054885   Other
          Pauljensenia      3-Indoxyl sulfate             Indole         0.053200   Other
             CCUG-7971             Riboflavin Vitamin_Riboflavin         0.048444   Other
           Clostridium             Riboflavin Vitam

## 11 · Pipeline Summary Report

In [12]:
try:
    pre = load_pickle(INTER_DIR / "preprocessed_data.pkl")
    n_datasets = len(pre.get("species_clr", {}))
except Exception:
    n_datasets = "N/A"

# NB06-1: correct key is "corr_all" (not "corr_sig") — matches NB02 save cell
corr_sig = ana.get("corr_all", pd.DataFrame())

print("=" * 65)
print("PIPELINE SUMMARY — Metabolite–Metagenomics Correlation Study")
print("=" * 65)
print(f"  NB01: {n_datasets} datasets preprocessed")
print(f"  NB02: {len(corr_sig):,} significant species–metabolite pairs (Spearman)")
print(f"  NB03: {len(targets)} metabolite targets benchmarked (top-50 dysregulated)")
print(f"  NB03: Top-50 SHAP species for GutSMASH targeting")

med_df   = med.get("mediation_df", pd.DataFrame())
cent_df  = med.get("centrality_df", pd.DataFrame())
print(f"  NB05: {len(med_df)} bootstrap mediation pairs tested")
print(f"  NB05: {len(cent_df)} nodes in centrality analysis")

n_validated = int(shap_sub["gutsmash_positive"].sum())  if not shap_sub.empty else "N/A"
n_novel_    = int((~shap_sub["gutsmash_positive"]).sum()) if not shap_sub.empty else "N/A"
print(f"  NB06: SHAP+ / GutSMASH+ (validated) : {n_validated}")
print(f"  NB06: SHAP+ / GutSMASH- (novel)      : {n_novel_}")

print()


Loaded: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl
PIPELINE SUMMARY — Metabolite–Metagenomics Correlation Study
  NB01: 6 datasets preprocessed
  NB02: 14,656 significant species–metabolite pairs (Spearman)
  NB03: 50 metabolite targets benchmarked (top-50 dysregulated)
  NB03: Top-50 SHAP species for GutSMASH targeting
  NB05: 30 bootstrap mediation pairs tested
  NB05: 334 nodes in centrality analysis
  NB06: SHAP+ / GutSMASH+ (validated) : 24
  NB06: SHAP+ / GutSMASH- (novel)      : 117



## 12–14 · Enhanced Visualisation Suite

Three complementary views of the SHAP × GutSMASH evidence space:

| Cell | Figure | What it shows |
|---|---|---|
| 12 | `nb06_shap_gutsmash_scatter.png` | SHAP importance vs GutSMASH confidence (continuous scatter) + stacked genus breakdown |
| 13 | `nb06_species_metabolite_heatmap.png` | Top-50 species × all GutSMASH classes — full producer catalogue |
| 14 | `nb06_bipartite_network.png` | Bipartite genera ↔ metabolite class network — edge colour = BGC evidence level |

**Interpretation note:** GutSMASH is a *specialised metabolite* mining tool. Primary metabolism targets (NAD+, G6P, Fumarate, TCA intermediates) are correctly classified as `primary_metabolism` and excluded from the novel-candidate pool — not because they lack evidence but because they are outside GutSMASH's design scope.

In [13]:
# ── 12 · SHAP × GutSMASH Continuous Evidence Scatter + Genus Breakdown ────────
# Replaces the 2×2 binary confusion matrix:
#   Left:  scatter X=SHAP importance, Y=GutSMASH confidence (0/0.5/1.0)
#   Right: stacked horizontal bar by genus showing evidence tier breakdown

if not shap_sub.empty:
    import numpy as _np2
    _rng = _np2.random.default_rng(42)

    _score_map = {"species": 1.0, "genus": 0.5, "novel": 0.0}
    _sc_df = shap_sub.copy()
    _sc_df["gutsmash_score"] = _sc_df["match_level"].map(_score_map).fillna(0.0)
    _sc_df["y_jitter"] = _sc_df["gutsmash_score"] + _rng.normal(0, 0.022, len(_sc_df))

    _PW_CMAP = {"Polyamine": "#9C27B0", "SCFA": "#2196F3", "Amino Acid": "#FF9800",
                "Indole / Tryptophan": "#009688", "Other": "#78909C"}
    _sc_df["pw_color"] = _sc_df["pathway"].map(_PW_CMAP).fillna("#78909C")

    fig_s, (ax_sc, ax_bar) = plt.subplots(1, 2, figsize=(15, 7),
                                            gridspec_kw={"width_ratios": [1.7, 1]})

    # ── Scatter ──────────────────────────────────────────────────────────────
    for _pw, _grp in _sc_df.groupby("pathway"):
        ax_sc.scatter(_grp["shap_importance"], _grp["y_jitter"],
                      c=_grp["pw_color"], s=55, alpha=0.82,
                      label=_pw, edgecolors="white", linewidths=0.4, zorder=3)

    # Annotate top-6 by SHAP
    for _, _r in _sc_df.nlargest(6, "shap_importance").iterrows():
        ax_sc.annotate(
            f"{_r['genus']}\n({_r['metabolite_name'][:18]})",
            xy=(_r["shap_importance"], _r["y_jitter"]),
            xytext=(_r["shap_importance"] + _sc_df["shap_importance"].max() * 0.05,
                    _r["y_jitter"] + 0.04),
            fontsize=6.5, ha="left",
            arrowprops=dict(arrowstyle="-", color="#bbb", lw=0.6),
            zorder=5
        )

    ax_sc.set_yticks([0.0, 0.5, 1.0])
    ax_sc.set_yticklabels(
        ["Novel (0.0)\nSHAP+ / GutSMASH–",
         "Genus-confirmed (0.5)\nSHAP+ / GutSMASH genus",
         "Species-confirmed (1.0)\nSHAP+ / GutSMASH species"],
        fontsize=8
    )
    ax_sc.set_xlabel("SHAP Importance (top-50 species)", fontsize=9)
    ax_sc.set_title("SHAP Importance vs GutSMASH Evidence Confidence\n(Y jittered for readability)",
                    fontsize=9.5, fontweight="bold")
    ax_sc.axhline(0.25, ls="--", lw=0.6, color="#ccc")
    ax_sc.axhline(0.75, ls="--", lw=0.6, color="#ccc")
    ax_sc.legend(title="Pathway", fontsize=7, title_fontsize=8, loc="upper right", framealpha=0.9)
    ax_sc.set_ylim(-0.20, 1.30)
    ax_sc.grid(axis="x", alpha=0.25)
    ax_sc.spines[["top", "right"]].set_visible(False)

    # ── Stacked bar by genus ─────────────────────────────────────────────────
    _gc = (
        _sc_df.groupby(["genus", "match_level"]).size()
        .unstack(fill_value=0)
        .reindex(columns=["species", "genus", "novel"], fill_value=0)
    )
    _gc_sorted = _gc.loc[_gc.sum(axis=1).sort_values(ascending=True).index]
    _BAR_COLS = {"species": "#4CAF50", "genus": "#FF9800", "novel": "#F44336"}
    _bots = _np2.zeros(len(_gc_sorted))
    for _lv, _clr in _BAR_COLS.items():
        if _lv in _gc_sorted.columns:
            _vals = _gc_sorted[_lv].values
            ax_bar.barh(range(len(_gc_sorted)), _vals, left=_bots,
                        color=_clr, label=_lv.capitalize(), alpha=0.88)
            _bots += _vals
    ax_bar.set_yticks(range(len(_gc_sorted)))
    ax_bar.set_yticklabels(_gc_sorted.index, fontsize=7.5, fontstyle="italic")
    ax_bar.set_xlabel("# (species, metabolite) pairs", fontsize=9)
    ax_bar.set_title("Evidence breakdown\nby genus", fontsize=9.5, fontweight="bold")
    ax_bar.legend(title="Match level", fontsize=7, title_fontsize=8,
                  loc="lower right", framealpha=0.9)
    ax_bar.spines[["top", "right"]].set_visible(False)

    fig_s.suptitle("GutSMASH Evidence Quality — SHAP-predicted Producers (top-50 species)",
                   fontsize=11, fontweight="bold")
    plt.tight_layout()
    savefig(fig_s, "gutsmash", "nb06_shap_gutsmash_scatter.png")
    plt.show()
    print(f"Scatter/bar saved: nb06_shap_gutsmash_scatter.png  ({len(_sc_df)} pairs)")
else:
    print("No SHAP–GutSMASH data for scatter. Re-run cells 6–7 first.")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_shap_gutsmash_scatter.png
Scatter/bar saved: nb06_shap_gutsmash_scatter.png  (141 pairs)


In [14]:
# ── 13 · Species × GutSMASH Metabolite Class Heatmap (Producer Catalogue) ────
# Rows = top-50 SHAP species (sorted by total SHAP, genus label shown)
# Cols = GutSMASH metabolite classes
# Cell colour = SHAP importance (0 = grey)
# Marker overlay = BGC evidence level (● species / ◐ genus / ✗ novel)

from matplotlib.lines import Line2D as _L2D

if not shap_long.empty and "gutsmash_class" in shap_long.columns:
    _hm_df = shap_long[shap_long["gutsmash_class"].notna()].copy()

    # Pivot: species × metabolite class, value = max SHAP importance
    _piv = (
        _hm_df.groupby(["species", "gutsmash_class"])["shap_importance"]
        .max().unstack(fill_value=0.0)
    )
    _piv = _piv.reindex(top50_species, fill_value=0.0).fillna(0.0)

    # Match level pivot
    if not shap_sub.empty:
        _ml_map = {"species": 1.0, "genus": 0.5, "novel": 0.0}
        _piv_ml = (
            shap_sub.groupby(["species", "gutsmash_class"])["match_level"]
            .apply(lambda x: max((_ml_map.get(v, 0.0) for v in x), default=0.0))
            .unstack(fill_value=float("nan"))
        )
        _piv_ml = _piv_ml.reindex(index=_piv.index, columns=_piv.columns, fill_value=float("nan"))
    else:
        import numpy as _np3
        _piv_ml = pd.DataFrame(_np3.nan, index=_piv.index, columns=_piv.columns)

    # Sort rows by total SHAP importance
    _row_ord = _piv.sum(axis=1).sort_values(ascending=False).index
    _piv    = _piv.loc[_row_ord]
    _piv_ml = _piv_ml.loc[_row_ord]

    _genus_labels = [extract_genus(sp) for sp in _piv.index]

    _fh = max(10, len(_piv) * 0.30)
    _fw = max(12, len(_piv.columns) * 0.90)
    fig_hm, ax_hm = plt.subplots(figsize=(_fw, _fh))

    import numpy as _np4
    _im = ax_hm.imshow(_piv.values, cmap="YlOrRd", aspect="auto",
                       vmin=0, interpolation="nearest")

    # Overlay match markers
    for _i in range(len(_piv)):
        for _j in range(len(_piv.columns)):
            _sv = _piv.values[_i, _j]
            _mv = _piv_ml.values[_i, _j]
            if _sv > 0:
                if not _np4.isnan(_mv):
                    if _mv >= 1.0:
                        _mk, _ms, _mc = "o", 7, "#1B5E20"   # species ● dark green
                    elif _mv >= 0.5:
                        _mk, _ms, _mc = "o", 6, "#F57F17"   # genus  ● amber
                    else:
                        _mk, _ms, _mc = "x", 6, "#B71C1C"   # novel  ✗ red
                    ax_hm.plot(_j, _i, marker=_mk, markersize=_ms, color=_mc,
                               markeredgewidth=0.9, zorder=4)

    ax_hm.set_xticks(range(len(_piv.columns)))
    ax_hm.set_xticklabels(_piv.columns, rotation=45, ha="right", fontsize=8)
    ax_hm.set_yticks(range(len(_genus_labels)))
    ax_hm.set_yticklabels(_genus_labels, fontsize=7.5, fontstyle="italic")
    ax_hm.set_xlabel("GutSMASH Metabolite Class", fontsize=9)
    ax_hm.set_ylabel("Species (genus label, sorted by SHAP)", fontsize=9)
    ax_hm.set_title(
        "SHAP Importance × GutSMASH Metabolite Class — Top-50 Predicted Producers\n"
        "● dark-green = species-level BGC   ● amber = genus-level   ✗ red = novel",
        fontsize=9.5, fontweight="bold"
    )
    _cb = plt.colorbar(_im, ax=ax_hm, shrink=0.45, pad=0.02)
    _cb.set_label("Max SHAP importance", fontsize=8)

    _leg_elems = [
        _L2D([0],[0], marker="o", color="w", markerfacecolor="#1B5E20", markersize=9, label="Species-level BGC"),
        _L2D([0],[0], marker="o", color="w", markerfacecolor="#F57F17", markersize=9, label="Genus-level BGC"),
        _L2D([0],[0], marker="x", color="#B71C1C", markersize=9, markeredgewidth=1.2, label="Novel candidate"),
    ]
    ax_hm.legend(handles=_leg_elems, loc="lower right", fontsize=8, framealpha=0.9)
    plt.tight_layout()
    savefig(fig_hm, "gutsmash", "nb06_species_metabolite_heatmap.png")
    plt.show()
    print(f"Heatmap: {len(_piv)} species × {len(_piv.columns)} classes | "
          f"non-zero cells: {(_piv.values > 0).sum()}")
else:
    print("Insufficient data for species × metabolite class heatmap.")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_species_metabolite_heatmap.png
Heatmap: 50 species × 15 classes | non-zero cells: 126


In [15]:
# ── 14 · Bipartite Species–Metabolite Network (GutSMASH Evidence Edges) ────────
# Left  nodes: genera (size ∝ total SHAP importance, blue circles)
# Right nodes: GutSMASH metabolite classes (size ∝ n producer genera, orange diamonds)
# Edges: width ∝ SHAP importance, colour = GutSMASH evidence level

try:
    import networkx as nx
    _nx_ok = True
except ImportError:
    _nx_ok = False
    print("networkx not installed — run: pip install networkx")

if _nx_ok and not shap_long.empty and "gutsmash_class" in shap_long.columns:
    import numpy as _np5
    _net = shap_long[shap_long["gutsmash_class"].notna()].copy()
    if not shap_sub.empty:
        _ml_lkp = (
            shap_sub.groupby(["genus","gutsmash_class"])["match_level"]
            .apply(lambda x: x.mode()[0] if len(x) else "novel")
            .to_dict()
        )
        _net["match_level"] = _net.apply(
            lambda r: _ml_lkp.get((r["genus"], r["gutsmash_class"]), "novel"), axis=1)
    else:
        _net["match_level"] = "novel"

    _edges = (
        _net.groupby(["genus","gutsmash_class"])
        .agg(total_shap=("shap_importance","sum"),
             match=("match_level", lambda x: x.mode()[0] if len(x) else "novel"))
        .reset_index()
    )

    G_net = nx.Graph()
    _gs_by_genus = _net.groupby("genus")["shap_importance"].sum()
    for _g, _s in _gs_by_genus.items():
        G_net.add_node(_g, ntype="genus", shap=float(_s))
    _cls_cnt = _net.groupby("gutsmash_class")["genus"].nunique()
    for _mc, _n in _cls_cnt.items():
        G_net.add_node(_mc, ntype="met", cnt=int(_n))

    _ECOL = {"species": "#4CAF50", "genus": "#FF9800", "novel": "#F44336"}
    _max_e = _edges["total_shap"].max() if len(_edges) else 1.0
    for _, _r in _edges.iterrows():
        G_net.add_edge(_r["genus"], _r["gutsmash_class"],
                       weight=float(_r["total_shap"]),
                       match=_r["match"], color=_ECOL.get(_r["match"], "#999"))

    fig_net, ax_net = plt.subplots(figsize=(15, 10))

    _gn = [n for n,d in G_net.nodes(data=True) if d.get("ntype")=="genus"]
    _mn = [n for n,d in G_net.nodes(data=True) if d.get("ntype")=="met"]
    _gn_sorted = sorted(_gn, key=lambda n: G_net.nodes[n].get("shap",0), reverse=True)
    _mn_sorted = sorted(_mn)

    _pos = {}
    for _i, _g in enumerate(_gn_sorted):
        _pos[_g] = _np5.array([-1.0, _i/max(1,len(_gn_sorted)-1)])
    for _j, _m in enumerate(_mn_sorted):
        _pos[_m] = _np5.array([1.0, _j/max(1,len(_mn_sorted)-1)])

    for _u, _v, _d in G_net.edges(data=True):
        _lw = 0.8 + 7.0 * _d["weight"] / _max_e
        _x = [_pos[_u][0], _pos[_v][0]]
        _y = [_pos[_u][1], _pos[_v][1]]
        ax_net.plot(_x, _y, color=_d["color"], linewidth=_lw, alpha=0.60, zorder=1)

    _mxs = max((d.get("shap",0.001) for _,d in G_net.nodes(data=True) if d.get("ntype")=="genus"), default=0.001)
    for _n in _gn_sorted:
        _x,_y = _pos[_n]; _sz = 200+700*G_net.nodes[_n].get("shap",0)/_mxs
        ax_net.scatter(_x, _y, s=_sz, c="#1565C0", zorder=3, alpha=0.85, edgecolors="white", lw=0.8)
        ax_net.text(_x-0.04, _y, _n, ha="right", va="center", fontsize=7.5, fontstyle="italic")
    _mxc = max((d.get("cnt",1) for _,d in G_net.nodes(data=True) if d.get("ntype")=="met"), default=1)
    for _n in _mn_sorted:
        _x,_y = _pos[_n]; _sz = 120+450*G_net.nodes[_n].get("cnt",1)/_mxc
        ax_net.scatter(_x, _y, s=_sz, c="#E65100", zorder=3, alpha=0.85,
                       edgecolors="white", lw=0.8, marker="D")
        ax_net.text(_x+0.04, _y, _n, ha="left", va="center", fontsize=7.5)

    from matplotlib.lines import Line2D as _L2D2
    _le = [
        _L2D2([0],[0], color="#4CAF50", lw=2.5, label="Species-confirmed BGC"),
        _L2D2([0],[0], color="#FF9800", lw=2.5, label="Genus-confirmed BGC"),
        _L2D2([0],[0], color="#F44336", lw=2.5, label="Novel candidate"),
        _L2D2([0],[0], marker="o", color="w", markerfacecolor="#1565C0", ms=10, label="Genus (size=SHAP)"),
        _L2D2([0],[0], marker="D", color="w", markerfacecolor="#E65100", ms=10, label="Met class (size=n_genera)"),
    ]
    ax_net.legend(handles=_le, loc="lower center", ncol=3, fontsize=8,
                  bbox_to_anchor=(0.5, -0.04), framealpha=0.9)
    ax_net.set_xlim(-1.45, 1.45); ax_net.set_ylim(-0.08, 1.08); ax_net.axis("off")
    ax_net.set_title(
        "Bipartite Network: SHAP-predicted Producers ↔ GutSMASH Metabolite Classes\n"
        "Edge width = SHAP importance  |  Edge colour = BGC evidence level",
        fontsize=10, fontweight="bold"
    )
    plt.tight_layout()
    savefig(fig_net, "gutsmash", "nb06_bipartite_network.png")
    plt.show()
    print(f"Network: {len(_gn)} genera, {len(_mn)} metabolite classes, {G_net.number_of_edges()} edges")
else:
    if not _nx_ok:
        pass  # already printed warning
    else:
        print("Insufficient data for bipartite network.")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_bipartite_network.png
Network: 38 genera, 15 metabolite classes, 116 edges


## 15–17 · Literature-Based Validation

Automated PubMed evidence mining assigns each (genus, metabolite) pair a tier:

| Tier | Label | Criteria |
|---|---|---|
| 0 | No evidence | 0 PubMed papers |
| 1 | Bioinformatic | Genomic prediction / annotation papers only |
| 2 | In vitro | Pure culture or metabolomics experimental evidence |
| 3 | In vivo | Germ-free mouse or animal colonisation study |
| 4 | Clinical | Human patient cohort or CRC-specific study |

Cache stored at `Results/tables/literature_evidence_cache.json` — re-runs are instant.

**Cell 17 (Holy Trinity)** combines all three evidence legs into a single publication figure. MICOM panel (③) auto-populates once `micom_flux_summary_staged.csv` is generated by NB09.

In [16]:
# ── 15 · Automated PubMed Evidence Mining for Novel Candidates ────────────────
# For each (genus, metabolite class) pair in novel + GutSMASH-matched sets,
# query NCBI PubMed E-utilities to assign a literature evidence tier:
#   Tier 0: No papers found
#   Tier 1: Bioinformatic / genomic prediction only
#   Tier 2: In vitro (pure culture / metabolomics)
#   Tier 3: In vivo (germ-free mouse / animal colonization)
#   Tier 4: Clinical (human patient cohort / CRC study)
#
# Results cached to table_dir/literature_evidence_cache.json — re-runs are instant.
# Rate limit: ≤3 requests/second (NCBI E-utilities without API key).

import requests, time, sys
import xml.etree.ElementTree as _ET
import json as _jl

_NCBI_BASE  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
_NCBI_EMAIL = "andnair@gmail.com"
_NCBI_TOOL  = "crc_microbiome_pipeline"
_CACHE_PATH = TABLE_DIR / "literature_evidence_cache.json"
_LIT_PATH   = TABLE_DIR / "literature_evidence_scores.csv"

# Load existing cache
_lit_cache = {}
if _CACHE_PATH.exists():
    with open(_CACHE_PATH) as _f:
        _lit_cache = _jl.load(_f)
    print(f"Cache loaded: {len(_lit_cache)} previously queried pairs")

def _pubmed_n(query: str) -> int:
    """Return PubMed article count; uses local cache to avoid re-querying."""
    _key = query.strip().lower()
    if _key in _lit_cache:
        return _lit_cache[_key]
    for _attempt in range(3):
        try:
            _r = requests.get(
                f"{_NCBI_BASE}/esearch.fcgi",
                params={"db":"pubmed","term":query,"rettype":"count",
                        "tool":_NCBI_TOOL,"email":_NCBI_EMAIL},
                timeout=12
            )
            if _r.status_code == 200:
                _root = _ET.fromstring(_r.text)
                _el   = _root.find("Count")
                _cnt  = int(_el.text) if _el is not None else 0
                _lit_cache[_key] = _cnt
                time.sleep(0.35)   # NCBI: ≤3 req/s
                return _cnt
        except Exception:
            if _attempt < 2: time.sleep(1.5)
    _lit_cache[_key] = 0
    return 0

def _classify_tier(genus: str, metabolite: str) -> dict:
    """5-query PubMed tier classification for one (genus, metabolite) pair."""
    _g = genus.strip()
    _m = metabolite.strip().replace("SCFA_","").replace("BileAcid_","").replace("_"," ")
    _qs = {
        "clinical":    (f'"{_g}"[TIAB] AND "{_m}"[TIAB] AND '
                        f'("colorectal cancer"[TIAB] OR "CRC"[TIAB] OR "patient"[TIAB] OR "cohort"[TIAB])'),
        "in_vivo":     (f'"{_g}"[TIAB] AND "{_m}"[TIAB] AND '
                        f'("germ-free"[TIAB] OR "mouse"[TIAB] OR "in vivo"[TIAB] OR "colonization"[TIAB])'),
        "in_vitro":    (f'"{_g}"[TIAB] AND "{_m}"[TIAB] AND '
                        f'("in vitro"[TIAB] OR "pure culture"[TIAB] OR "fermentation"[TIAB])'),
        "bioinformatic":(f'"{_g}"[TIAB] AND "{_m}"[TIAB] AND '
                        f'("prediction"[TIAB] OR "bioinformatic"[TIAB] OR "genome"[TIAB])'),
        "general":     f'"{_g}"[TIAB] AND ("{_m} production"[TIAB] OR "{_m} biosynthesis"[TIAB] OR "{_m}"[TIAB] AND "gut"[TIAB])',
    }
    _cnts = {_k: _pubmed_n(_q) for _k, _q in _qs.items()}
    if   _cnts["clinical"]     >= 1: _tier = 4
    elif _cnts["in_vivo"]      >= 1: _tier = 3
    elif _cnts["in_vitro"]     >= 1: _tier = 2
    elif _cnts["bioinformatic"]>= 1 or _cnts["general"] >= 2: _tier = 1
    elif _cnts["general"]      >= 1: _tier = 1
    else:                            _tier = 0
    return {"tier": _tier, "counts": _cnts}

# ── Build mining list: novel candidates + GutSMASH-validated (for comparison) ─
if not shap_long.empty and "gutsmash_class" in shap_long.columns:
    _mine_df = shap_long[shap_long["gutsmash_class"].notna()].copy()
    _pairs   = (
        _mine_df[["genus","gutsmash_class","metabolite_name"]]
        .drop_duplicates()
        .query("genus.str.len() > 4")    # exclude short GTDB bin IDs
        .reset_index(drop=True)
    )

    print(f"Mining {len(_pairs)} (genus, metabolite) pairs for PubMed evidence ...")
    print("  Cached pairs will be instant. First run ≈ 2–5 min for 40+ pairs.\n")

    _lit_rows = []
    for _idx, _row in _pairs.iterrows():
        _res = _classify_tier(_row["genus"], _row["metabolite_name"])
        _lit_rows.append({
            "genus":           _row["genus"],
            "gutsmash_class":  _row["gutsmash_class"],
            "metabolite_name": _row["metabolite_name"],
            "tier":            _res["tier"],
            "n_general":       _res["counts"]["general"],
            "n_in_vitro":      _res["counts"]["in_vitro"],
            "n_in_vivo":       _res["counts"]["in_vivo"],
            "n_clinical":      _res["counts"]["clinical"],
        })
        sys.stdout.write(f"\r  {len(_lit_rows)}/{len(_pairs)} complete  "); sys.stdout.flush()

    print()
    lit_scores_df = pd.DataFrame(_lit_rows)

    # Merge SHAP + GutSMASH match info
    if not shap_sub.empty:
        _merge_src = (
            shap_sub.groupby(["genus","gutsmash_class"])
            .agg(shap_importance=("shap_importance","max"),
                 match_level=("match_level","first"),
                 gutsmash_positive=("gutsmash_positive","first"))
            .reset_index()
        )
        lit_scores_df = lit_scores_df.merge(_merge_src, on=["genus","gutsmash_class"], how="left")
    else:
        lit_scores_df["shap_importance"]  = float("nan")
        lit_scores_df["match_level"]      = "novel"
        lit_scores_df["gutsmash_positive"]= False

    # Save cache + results
    with open(_CACHE_PATH, "w") as _fc:
        _jl.dump(_lit_cache, _fc, indent=2)
    lit_scores_df.to_csv(_LIT_PATH, index=False)

    _TLABELS = {0:"No evidence", 1:"Bioinformatic", 2:"In vitro", 3:"In vivo", 4:"Clinical"}
    print(f"\nLiterature mining complete: {len(lit_scores_df)} pairs")
    print("Tier distribution:")
    for _t in range(5):
        _n = (lit_scores_df["tier"] == _t).sum()
        print(f"  Tier {_t} ({_TLABELS[_t]}): {_n}")
    print(f"\nCache: {len(_lit_cache)} entries saved → {_CACHE_PATH.name}")
    print(f"Table: {_LIT_PATH.name}")
else:
    lit_scores_df = pd.DataFrame()
    print("No GutSMASH-mapped SHAP data — skipping PubMed mining.")


Mining 134 (genus, metabolite) pairs for PubMed evidence ...
  Cached pairs will be instant. First run ≈ 2–5 min for 40+ pairs.

  134/134 complete  

Literature mining complete: 134 pairs
Tier distribution:
  Tier 0 (No evidence): 88
  Tier 1 (Bioinformatic): 6
  Tier 2 (In vitro): 4
  Tier 3 (In vivo): 7
  Tier 4 (Clinical): 29

Cache: 670 entries saved → literature_evidence_cache.json
Table: literature_evidence_scores.csv


In [17]:
# ── 16 · Literature Evidence Heatmap + Novel Candidate Forest Plot ─────────────
# 16a: Genus × metabolite class heatmap coloured by max literature tier
#       Cell annotation = paper count (general PubMed query)
#       Left bar = GutSMASH support (green = confirmed, red = novel)
# 16b: Forest plot — novel candidates ranked by composite score SHAP × lit-tier

_TLABELS = {0:"No evidence", 1:"Bioinformatic", 2:"In vitro", 3:"In vivo", 4:"Clinical"}
_TCOLS   = {0:"#EEEEEE",     1:"#FFF9C4",       2:"#FFE082",  3:"#81D4FA",  4:"#A5D6A7"}

if "lit_scores_df" in dir() and not lit_scores_df.empty:
    import matplotlib.colors as _mcol
    import matplotlib.gridspec as _mgs
    import numpy as _np6

    # ── 16a — Evidence tier heatmap ──────────────────────────────────────────
    _pt = (
        lit_scores_df
        .groupby(["genus","gutsmash_class"])["tier"]
        .max().unstack(fill_value=0)
    )
    # Sort genera by max SHAP
    if "shap_importance" in lit_scores_df.columns:
        _gord = (
            lit_scores_df.groupby("genus")["shap_importance"].max()
            .sort_values(ascending=False).index
        )
        _pt = _pt.reindex([_g for _g in _gord if _g in _pt.index], fill_value=0)

    _gs_pos = (
        lit_scores_df.groupby("genus")["gutsmash_positive"].any()
        if "gutsmash_positive" in lit_scores_df.columns
        else pd.Series(False, index=_pt.index)
    )

    _tier_cmap = _mcol.ListedColormap([_TCOLS[i] for i in range(5)])
    _tier_norm = _mcol.BoundaryNorm([-0.5,0.5,1.5,2.5,3.5,4.5], ncolors=5)

    _figH = max(8, len(_pt)*0.55)
    _figW = max(12, len(_pt.columns)*1.0)
    fig_ev = plt.figure(figsize=(_figW, _figH))
    _gs2   = _mgs.GridSpec(1, 3, width_ratios=[0.09,1,0.06], wspace=0.03, figure=fig_ev)
    _ax_a  = fig_ev.add_subplot(_gs2[0,0])
    _ax_h2 = fig_ev.add_subplot(_gs2[0,1])
    _ax_cb = fig_ev.add_subplot(_gs2[0,2])

    # Row annotation bar
    for _i2, _g2 in enumerate(_pt.index):
        _ax_a.barh(_i2, 1, color="#4CAF50" if _gs_pos.get(_g2,False) else "#F44336", alpha=0.8)
    _ax_a.set_yticks(range(len(_pt)))
    _ax_a.set_yticklabels(_pt.index, fontsize=8, fontstyle="italic")
    _ax_a.set_xticks([0.5]); _ax_a.set_xticklabels(["GS"], fontsize=7)
    _ax_a.set_xlim(0,1); _ax_a.set_ylim(-0.5,len(_pt)-0.5)
    _ax_a.invert_yaxis(); _ax_a.yaxis.tick_right()

    _im2 = _ax_h2.imshow(_pt.values, cmap=_tier_cmap, norm=_tier_norm,
                         aspect="auto", interpolation="nearest")

    # Cell paper-count annotations
    if "n_general" in lit_scores_df.columns:
        _cnt_piv = (
            lit_scores_df
            .groupby(["genus","gutsmash_class"])["n_general"].max()
            .unstack(fill_value=0)
            .reindex(index=_pt.index, columns=_pt.columns, fill_value=0)
        )
        for _i3 in range(len(_pt)):
            for _j3 in range(len(_pt.columns)):
                _n3 = int(_cnt_piv.values[_i3,_j3])
                _t3 = int(_pt.values[_i3,_j3])
                if _n3 > 0:
                    _ax_h2.text(_j3, _i3, str(_n3), ha="center", va="center",
                                fontsize=7.5, color="#FFF" if _t3 >= 3 else "#333")

    _ax_h2.set_xticks(range(len(_pt.columns)))
    _ax_h2.set_xticklabels(_pt.columns, rotation=45, ha="right", fontsize=8)
    _ax_h2.set_yticks([]); _ax_h2.set_xlabel("GutSMASH Metabolite Class", fontsize=9)
    _ax_h2.set_title(
        "Literature Evidence Tier per (Genus × Metabolite Class)\n"
        "Cell = # PubMed papers  |  GS bar = GutSMASH support (green=yes, red=novel)",
        fontsize=9, fontweight="bold"
    )
    plt.colorbar(_im2, cax=_ax_cb)
    _ax_cb.set_yticks(range(5))
    _ax_cb.set_yticklabels([_TLABELS[i] for i in range(5)], fontsize=7)
    _ax_cb.set_ylabel("Literature Evidence Tier", fontsize=7, rotation=-90, labelpad=10)
    savefig(fig_ev, "gutsmash", "nb06_literature_evidence_heatmap.png")
    plt.show()
    print("Fig 16a saved: nb06_literature_evidence_heatmap.png")

    # ── 16b — Forest plot: novel candidates ranked by composite score ─────────
    if "novel" in dir() and not novel.empty and "tier" in lit_scores_df.columns:
        _for_df = novel.merge(
            lit_scores_df[["genus","gutsmash_class","tier","n_general","n_in_vitro","n_in_vivo"]],
            on=["genus","gutsmash_class"], how="left"
        ).fillna({"tier":0,"n_general":0,"n_in_vitro":0,"n_in_vivo":0})
        _for_df["tier"]            = _for_df["tier"].astype(int)
        _for_df["composite_score"] = _for_df["shap_importance"] * (1 + _for_df["tier"] / 4.0)
        _for_df = _for_df.sort_values("composite_score", ascending=True)

        _labels_f = [f"{_r['genus']}\n({_r['metabolite_name'][:20]})"
                     for _, _r in _for_df.iterrows()]
        _bar_cols_f = [_TCOLS[int(_t)] for _t in _for_df["tier"]]

        fig_fp, ax_fp = plt.subplots(figsize=(10, max(5, len(_for_df)*0.52)))
        ax_fp.barh(range(len(_for_df)), _for_df["composite_score"],
                   color=_bar_cols_f, edgecolor="#888", linewidth=0.5, height=0.7)
        _xmax = _for_df["composite_score"].max() if len(_for_df) else 1.0
        for _i4, (_, _r4) in enumerate(_for_df.iterrows()):
            _np4 = int(_r4.get("n_general", 0))
            if _np4 > 0:
                ax_fp.text(_xmax * 0.02, _i4, f"{_np4} papers",
                           va="center", fontsize=7, color="#444")
        ax_fp.set_yticks(range(len(_labels_f)))
        ax_fp.set_yticklabels(_labels_f, fontsize=8)
        ax_fp.set_xlabel("Composite score  (SHAP × literature-tier boost)", fontsize=9)
        ax_fp.set_title(
            "Novel Producer Candidates — Ranked by SHAP × Literature Evidence\n"
            "Bar colour = highest literature evidence tier found",
            fontsize=10, fontweight="bold"
        )
        ax_fp.spines[["top","right"]].set_visible(False)
        from matplotlib.patches import Patch as _Pa
        ax_fp.legend(
            handles=[_Pa(color=_TCOLS[_t], label=f"Tier {_t}: {_TLABELS[_t]}") for _t in range(5)],
            loc="lower right", fontsize=8, title="Literature tier", title_fontsize=9
        )
        plt.tight_layout()
        savefig(fig_fp, "gutsmash", "nb06_novel_candidates_forest_plot.png")
        plt.show()
        print("Fig 16b saved: nb06_novel_candidates_forest_plot.png")
        print("\nTop novel candidates by composite score:")
        print(_for_df[["genus","metabolite_name","shap_importance","tier","n_general","composite_score"]]
              .sort_values("composite_score", ascending=False).head(10).to_string(index=False))
else:
    print("Literature evidence data not available — run cell 15 first.")


Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_literature_evidence_heatmap.png
Fig 16a saved: nb06_literature_evidence_heatmap.png
Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_novel_candidates_forest_plot.png
Fig 16b saved: nb06_novel_candidates_forest_plot.png

Top novel candidates by composite score:
                 genus        metabolite_name  shap_importance  tier  n_general  composite_score
            Aphodousia        trans-Cinnamate         0.159896     0          0         0.159896
          Scatavimonas             Pyridoxine         0.084577     0          0         0.084577
           Clostridium             Riboflavin         0.047149     3          8         0.082511
Erysipelatoclostridium            Glucuronate         0.056788     0          0         0.056788
          Pauljensenia Trimethylamine N-oxide         0.055671     0          0         0.055671
         Streptococcus            Glucuronate         0.036704

In [18]:
# ── 17 · Holy Trinity Integration: SHAP × GutSMASH × MICOM ─────────────────────
# Three-panel per metabolite:
#   Panel ①: SHAP importance (NB03 ML model — statistical predictor)
#   Panel ②: GutSMASH BGC evidence level (genomic capacity)
#   Panel ③: MICOM excretion flux (NB09 mechanistic FBA — if available)
#
# Requires: micom_flux_summary_staged.csv from NB09 (graceful fallback if absent).

_MICOM_PATH = TABLE_DIR / "micom_flux_summary_staged.csv"
_micom_ok   = _MICOM_PATH.exists()

if _micom_ok:
    micom_flux = pd.read_csv(_MICOM_PATH)
    print(f"MICOM flux loaded: {micom_flux.shape} — "
          f"{micom_flux['met_name'].nunique()} metabolites, "
          f"{micom_flux['stage'].nunique()} stages")
else:
    micom_flux = pd.DataFrame()
    print(f"MICOM flux summary not found: {_MICOM_PATH.name}")
    print("Run NB09 through cell-13 to enable panel ③.")

if not shap_sub.empty:
    _focal = shap_sub["metabolite_name"].unique().tolist()[:8]  # cap at 8 metabolites
    _n_row = len(_focal)

    if _n_row > 0:
        _MCOLORS = {"species":"#2E7D32", "genus":"#F9A825", "novel":"#C62828"}

        fig_t, _axes_t = plt.subplots(
            _n_row, 3, figsize=(15, max(5, 3.8*_n_row)),
            gridspec_kw={"wspace":0.38, "hspace":0.52}
        )
        if _n_row == 1: _axes_t = _axes_t.reshape(1, 3)

        for _ri, _met in enumerate(_focal):
            # ── Panel ①: SHAP ──────────────────────────────────────────────
            _ax1 = _axes_t[_ri, 0]
            _sm  = shap_sub[shap_sub["metabolite_name"]==_met].nlargest(8,"shap_importance")
            if len(_sm):
                _ax1.barh(range(len(_sm)), _sm["shap_importance"],
                          color=[_MCOLORS.get(_ml,"#999") for _ml in _sm["match_level"]],
                          alpha=0.85, edgecolor="white")
                _ax1.set_yticks(range(len(_sm)))
                _ax1.set_yticklabels(_sm["genus"].values, fontsize=7.5, fontstyle="italic")
                _ax1.invert_yaxis()
            _ax1.set_xlabel("Mean |SHAP|", fontsize=7)
            _ax1.set_title("① SHAP\n(NB03)", fontsize=8, color="#1565C0")
            _ax1.spines[["top","right"]].set_visible(False)

            # ── Panel ②: GutSMASH ──────────────────────────────────────────
            _ax2  = _axes_t[_ri, 1]
            _gsm  = shap_sub[shap_sub["metabolite_name"]==_met].copy()
            _gsag = (
                _gsm.groupby("genus")
                .agg(match=("match_level", lambda x: x.mode()[0] if len(x) else "novel"),
                     mshap=("shap_importance","max"))
                .reset_index()
                .sort_values("mshap", ascending=False).head(8)
            )
            if len(_gsag):
                _conf = [1.0 if _m=="species" else 0.5 if _m=="genus" else 0.1 for _m in _gsag["match"]]
                _ax2.barh(range(len(_gsag)), _conf,
                          color=[_MCOLORS.get(_m,"#999") for _m in _gsag["match"]],
                          alpha=0.85, edgecolor="white")
                _ax2.set_yticks(range(len(_gsag)))
                _ax2.set_yticklabels(
                    [f"{'●' if m=='species' else '◐' if m=='genus' else '○'} {g}"
                     for g,m in zip(_gsag["genus"],_gsag["match"])],
                    fontsize=7
                )
                _ax2.invert_yaxis()
                _ax2.set_xlim(0,1.2)
                _ax2.set_xticks([0.1,0.5,1.0])
                _ax2.set_xticklabels(["Novel","Genus","Species"], fontsize=6, rotation=30)
            _ax2.set_xlabel("BGC evidence level", fontsize=7)
            _ax2.set_title("② GutSMASH\n(BGC genomics)", fontsize=8, color="#E65100")
            _ax2.spines[["top","right"]].set_visible(False)

            # ── Panel ③: MICOM flux ────────────────────────────────────────
            _ax3 = _axes_t[_ri, 2]
            if _micom_ok and not micom_flux.empty and _met in micom_flux["met_name"].values:
                _mf = (
                    micom_flux[micom_flux["met_name"]==_met]
                    .groupby("taxon_full")["mean_flux"].mean()
                    .sort_values(ascending=False).head(8)
                )
                _ax3.barh(range(len(_mf)), _mf.values,
                          color="#7B1FA2", alpha=0.80, edgecolor="white")
                _ax3.set_yticks(range(len(_mf)))
                _ax3.set_yticklabels(_mf.index, fontsize=7, fontstyle="italic")
                _ax3.invert_yaxis()
                _ax3.set_xlabel("Mean flux\n(mmol/gDW/h)", fontsize=7)
            else:
                _ax3.text(0.5, 0.5, "NB09 not yet run\n(micom_flux_summary_staged.csv)",
                          transform=_ax3.transAxes, ha="center", va="center",
                          fontsize=8, color="#999", style="italic")
                _ax3.set_yticks([])
            _ax3.set_title("③ MICOM Flux\n(FBA mechanistic)", fontsize=8, color="#6A1B9A")
            _ax3.spines[["top","right"]].set_visible(False)

            # Metabolite name centred above all 3 panels
            _pl = _axes_t[_ri,0].get_position(); _pr = _axes_t[_ri,2].get_position()
            fig_t.text((_pl.x0+_pr.x1)/2, _pl.y1+0.009, _met,
                       ha="center", va="bottom", fontsize=9.5, fontweight="bold")

        # Column headers
        for _ci2, (_ct, _cc) in enumerate(zip(
                ["① SHAP Importance (NB03)","② GutSMASH BGC Support","③ MICOM Flux (NB09)"],
                ["#1565C0","#E65100","#6A1B9A"])):
            _pp = _axes_t[0,_ci2].get_position()
            fig_t.text((_pp.x0+_pp.x1)/2, _pp.y1+0.022, _ct,
                       ha="center", va="bottom", fontsize=9, color=_cc, fontweight="bold")

        from matplotlib.patches import Patch as _Pa2
        fig_t.legend(
            handles=[
                _Pa2(color="#2E7D32", label="Species-level BGC (GutSMASH)"),
                _Pa2(color="#F9A825", label="Genus-level BGC"),
                _Pa2(color="#C62828", label="Novel candidate"),
            ],
            loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5,0.0), frameon=False
        )
        fig_t.suptitle(
            "Holy Trinity Integration: Statistical (SHAP) × Genomic (GutSMASH) × Mechanistic (MICOM)",
            fontsize=10.5, fontweight="bold", y=1.005
        )
        fig_t.subplots_adjust(top=0.92, bottom=0.07)
        savefig(fig_t, "gutsmash", "nb06_holy_trinity_integration.png")
        plt.show()
        print(f"Holy Trinity figure saved for {_n_row} metabolites.")
    else:
        print("No focal metabolites for Holy Trinity plot.")
else:
    print("No SHAP–GutSMASH data. Run cells 6–7 first.")


MICOM flux loaded: (10432, 9) — 172 metabolites, 3 stages
Saved figure: C:\Users\andna\Documents\KI\Results\figures\gutsmash\nb06_holy_trinity_integration.png
Holy Trinity figure saved for 8 metabolites.


## Appendix · GutSMASH Installation & HPC Usage

### Local installation
```bash
git clone https://github.com/victoriapascal/gutsmash
cd gutsmash && pip install -e .
```

### Run on a single genome
```bash
python gutsmash/run_gutsmash.py \
    --cpus 8 \
    --output-dir results/gutsmash_runs/genome1 \
    --genefinding-tool prodigal \
    genome1.fna
```

### Submit SLURM array job (generated in Section 3)
```bash
sbatch results/slurm_scripts/submit_gutsmash_array.sh
```

### Online server (≤5 genomes)
https://gutsmash.bioinformatics.nl/

### Reference
Louwen et al. (2023) GutSMASH: Predicting specialized metabolic pathways  
from gut microbiome metagenomes. *mSystems*.